### CSV reading, cleaning and saving

Exploratory notebook for a single raw flight log (cells 2–8).
To clean a whole dataset, run `python src/preprocess.py --dataset data/dataset_1` from the repository root, or cell 9 below.

In [ ]:
from pathlib import Path
import sys
import os

parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))
print(sys.path)

In [ ]:
import pandas as pd

# Point this to one raw flight log (space-separated, first column is an unnamed index)
csv_file_path = '../data/dataset_1/training_data/<flight>.csv'

df = pd.read_csv(csv_file_path, sep=' ')

print("Original columns:")
print(df.columns.tolist())
print(f"\nFirst column name: '{df.columns[0]}'")

# Check if the first column is an index column (unnamed or numeric)
# Common patterns: 'Unnamed: 0', empty string, or numeric column name
first_col = df.columns[0]
is_index_col = (first_col.startswith('Unnamed:') or 
                first_col == '' or 
                (isinstance(first_col, str) and first_col.strip() == ''))

if is_index_col:
    print(f"\nDetected index column: '{first_col}'")
    # Drop the first column (the index column)
    df = df.drop(df.columns[0], axis=1)
    print("✓ Dropped the index column")
else:
    print(f"\nFirst column '{first_col}' appears to be a data column, not an index")

# Display the first few rows to verify
print("\nFirst few rows after cleaning:")
print(df.head())

print("\nColumn names after cleaning:")
print(df.columns.tolist())

In [ ]:
df.describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("DATA QUALITY SANITY CHECKS")
print("=" * 80)

# 1. Basic dataset info
print("\n1. DATASET OVERVIEW")
print("-" * 80)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 2. Data types check
print("\n2. DATA TYPES ANALYSIS")
print("-" * 80)
print("\nData types summary:")
dtype_counts = df.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} columns")

print("\nDetailed column data types:")
for col in df.columns:
    dtype = df[col].dtype
    print(f"  {col:40s} {str(dtype):15s}", end="")
    # Check if numeric
    if pd.api.types.is_numeric_dtype(df[col]):
        if dtype in ['float64', 'float32']:
            print(" ✓ float/double")
        elif dtype in ['int64', 'int32', 'int16', 'int8']:
            print(" ⚠ integer (not double)")
        else:
            print(f" ⚠ numeric but {dtype}")
    else:
        print(" ⚠ non-numeric")

# 3. Check for NaN values
print("\n3. MISSING VALUES (NaN) ANALYSIS")
print("-" * 80)
nan_counts = df.isna().sum()
nan_percentages = (nan_counts / len(df)) * 100

if nan_counts.sum() == 0:
    print("✓ No missing values found!")
else:
    print("Columns with missing values:")
    nan_df = pd.DataFrame({
        'Column': nan_counts[nan_counts > 0].index,
        'NaN Count': nan_counts[nan_counts > 0].values,
        'Percentage': nan_percentages[nan_counts > 0].values
    }).sort_values('NaN Count', ascending=False)
    print(nan_df.to_string(index=False))
    
    print(f"\nTotal missing values: {nan_counts.sum()}")
    print(f"Columns with NaN: {len(nan_counts[nan_counts > 0])} out of {len(df.columns)}")

# 4. Check for infinite values
print("\n4. INFINITE VALUES CHECK")
print("-" * 80)
inf_counts = {}
for col in df.select_dtypes(include=[np.number]).columns:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        inf_counts[col] = inf_count

if len(inf_counts) == 0:
    print("✓ No infinite values found!")
else:
    print("Columns with infinite values:")
    for col, count in inf_counts.items():
        print(f"  {col}: {count} infinite values")

# 5. Check for duplicate rows
print("\n5. DUPLICATE ROWS CHECK")
print("-" * 80)
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count} ({duplicate_count/len(df)*100:.2f}%)")
if duplicate_count > 0:
    print("⚠ Warning: Duplicate rows found!")

# 6. Check for constant columns (columns with only one unique value)
print("\n6. CONSTANT COLUMNS CHECK")
print("-" * 80)
constant_cols = []
for col in df.columns:
    unique_count = df[col].nunique()
    if unique_count <= 1:
        constant_cols.append(col)
        print(f"  ⚠ {col}: {unique_count} unique value(s)")

if len(constant_cols) == 0:
    print("✓ No constant columns found")

# 7. Check for columns with very few unique values (potential categorical)
print("\n7. LOW CARDINALITY COLUMNS (< 10 unique values)")
print("-" * 80)
low_cardinality = []
for col in df.columns:
    unique_count = df[col].nunique()
    if unique_count < 10 and unique_count > 1:
        low_cardinality.append((col, unique_count))
        print(f"  {col}: {unique_count} unique values")

if len(low_cardinality) == 0:
    print("No low cardinality columns found")

# 8. Numeric columns statistics
print("\n8. NUMERIC COLUMNS STATISTICS")
print("-" * 80)
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(f"Found {len(numeric_cols)} numeric columns")
    print("\nSummary statistics:")
    print(df[numeric_cols].describe())
    
    # Check for potential outliers (values beyond 3 standard deviations)
    print("\n9. POTENTIAL OUTLIERS (beyond 3 standard deviations)")
    print("-" * 80)
    outlier_info = []
    for col in numeric_cols:
        if df[col].notna().sum() > 0:  # Only check if column has non-null values
            mean = df[col].mean()
            std = df[col].std()
            if std > 0:  # Avoid division by zero
                outliers = ((df[col] - mean).abs() > 3 * std).sum()
                if outliers > 0:
                    outlier_info.append({
                        'Column': col,
                        'Outliers': outliers,
                        'Percentage': (outliers / len(df)) * 100,
                        'Mean': mean,
                        'Std': std
                    })
    
    if len(outlier_info) > 0:
        outlier_df = pd.DataFrame(outlier_info)
        print(outlier_df.to_string(index=False))
    else:
        print("✓ No significant outliers detected (using 3σ rule)")
else:
    print("No numeric columns found")

# 10. Check for mixed types in columns (the warning we saw earlier)
print("\n10. MIXED TYPE COLUMNS CHECK")
print("-" * 80)
mixed_type_cols = []
for col in df.columns:
    # Check if column has mixed types by trying to convert to numeric
    if df[col].dtype == 'object':
        numeric_count = pd.to_numeric(df[col], errors='coerce').notna().sum()
        if 0 < numeric_count < len(df):
            mixed_type_cols.append((col, numeric_count, len(df) - numeric_count))
            print(f"  ⚠ {col}: {numeric_count} numeric values, {len(df) - numeric_count} non-numeric")

if len(mixed_type_cols) == 0:
    print("✓ No mixed type columns detected")

# 11. Date/Time columns check
print("\n11. DATE/TIME COLUMNS CHECK")
print("-" * 80)
date_cols = ['date', 'time', 'time_of_day']
for col in date_cols:
    if col in df.columns:
        print(f"  {col}: {df[col].dtype}")
        if df[col].dtype == 'object':
            # Try to parse as datetime
            try:
                pd.to_datetime(df[col].iloc[0])
                print(f"    ✓ Appears to be datetime format")
            except:
                print(f"    ⚠ May not be valid datetime format")

# 12. Value ranges for key numeric columns
print("\n12. VALUE RANGES FOR KEY NUMERIC COLUMNS")
print("-" * 80)
key_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in 
            ['latitude', 'longitude', 'altitude', 'speed', 'force', 'energy', 'power'])]
if len(key_cols) > 0:
    for col in key_cols:
        if col in numeric_cols:
            min_val = df[col].min()
            max_val = df[col].max()
            print(f"  {col:40s} Range: [{min_val:12.4f}, {max_val:12.4f}]")

print("\n" + "=" * 80)
print("SANITY CHECKS COMPLETE")
print("=" * 80)


In [ ]:
df = df[df['kite_height'] >= 20]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

# Prepare time axis - convert to seconds from start if needed
if 'time' in df.columns:
    time_data = df['time'].values
    # If time is Unix timestamp, convert to seconds from start
    if time_data[0] > 1e9:  # Likely Unix timestamp
        time_axis = time_data - time_data[0]  # Seconds from start
        time_label = 'Time (seconds from start)'
    else:
        time_axis = time_data
        time_label = 'Time'
else:
    # Use index as time if no time column
    time_axis = df.index.values
    time_label = 'Sample index'

# --------------------------------------------------
# Select random 10-second window (optional)
# --------------------------------------------------
RANDOM_WINDOW_SECONDS = 120
USE_RANDOM_WINDOW = True  # set False to plot full data

if USE_RANDOM_WINDOW:
    t_min = time_axis.min()
    t_max = time_axis.max()

    if (t_max - t_min) > RANDOM_WINDOW_SECONDS:
        random_start = np.random.uniform(
            t_min,
            t_max - RANDOM_WINDOW_SECONDS
        )
        random_end = random_start + RANDOM_WINDOW_SECONDS

        time_mask = (time_axis >= random_start) & (time_axis <= random_end)

        time_axis = time_axis[time_mask]
        df = df.loc[time_mask].reset_index(drop=True)

        print(f"Plotting random window: {random_start:.2f} → {random_end:.2f} seconds")
    else:
        print("Data shorter than 10 seconds — plotting full range")

# --------------------------------------------------
# Unwrap yaw signals
# --------------------------------------------------
yaw_columns = [
    'kite_0_roll',
    'kite_0_pitch',
    'kite_0_yaw',
    'kite_1_roll',
    'kite_1_pitch',
    'kite_1_yaw',
    'kite_heading',
    'kite_course'
]

for col in yaw_columns:
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        # Heuristic: assume degrees if values exceed ±2π
        if df[col].abs().max() > 2 * np.pi:
            # Degrees → radians → unwrap → degrees
            yaw_rad = np.deg2rad(df[col].values)
            #df[col] = np.rad2deg(np.unwrap(yaw_rad))
            df[col] = np.unwrap(yaw_rad)
        else:
            # Already radians
            df[col] = np.unwrap(df[col].values)


# Define column groups for plotting
column_groups = {
    'Kite 0 Velocity Components': ['kite_0_vx', 'kite_0_vy', 'kite_0_vz'],
    'Kite 0 Position (Lat/Lon/Alt)': ['kite_0_latitude', 'kite_0_longitude', 'kite_0_altitude'],
    'Kite 0 Orientation (Roll/Pitch/Yaw)': ['kite_0_roll', 'kite_0_pitch', 'kite_0_yaw'],
    'Kite 1 Velocity Components': ['kite_1_vx', 'kite_1_vy', 'kite_1_vz'],
    'Kite 1 Position (Lat/Lon/Alt)': ['kite_1_latitude', 'kite_1_longitude', 'kite_1_altitude'],
    'Kite 1 Orientation (Roll/Pitch/Yaw)': ['kite_1_roll', 'kite_1_pitch', 'kite_1_yaw'],
    'Ground Tether Parameters': ['ground_tether_reelout_speed', 'ground_tether_length', 'ground_tether_force'],
    'Kite Position (East/North/Height)': ['kite_pos_east', 'kite_pos_north', 'kite_height'],
    'Kite Angles (Elevation/Azimuth/Distance)': ['kite_elevation', 'kite_azimuth'], #, 'kite_distance'],
    'Kite Navigation (Heading/Course)': ['kite_heading', 'kite_course'],
    'Kite Control (Actual vs Set)': ['kite_actual_steering', 'kite_actual_depower', 'kite_set_steering', 'kite_set_depower'],
    'Ground Mechanical (Energy/Power)': ['ground_mech_energy', 'ground_mech_power'],
    'Kite Rates': ['kite_turn_rate', 'kite_course_rate'],
}

# Single column plots (not grouped)
single_columns = ['flight_phase', 'flight_phase_index']

# Color palettes for different group sizes
colors_3 = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
colors_4 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Blue, Orange, Green, Red
colors_2 = ['#1f77b4', '#ff7f0e']  # Blue, Orange

# Create figure with subplots
n_groups = len(column_groups) + len(single_columns)
n_cols = 3
n_rows = (n_groups + n_cols - 1) // n_cols  # Ceiling division

fig = plt.figure(figsize=(20, 6 * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, hspace=0.4, wspace=0.3)

plot_idx = 0

# Plot grouped columns
for group_name, columns in column_groups.items():
    # Filter to only existing columns
    existing_cols = [col for col in columns if col in df.columns]
    
    if len(existing_cols) == 0:
        continue
    
    row = plot_idx // n_cols
    col = plot_idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    
    # Select appropriate colors
    if len(existing_cols) == 2:
        colors = colors_2
    elif len(existing_cols) == 3:
        colors = colors_3
    elif len(existing_cols) == 4:
        colors = colors_4
    else:
        colors = plt.cm.tab10(np.linspace(0, 1, len(existing_cols)))
    
    # Plot each column in the group
    for i, col_name in enumerate(existing_cols):
        if col_name in df.columns and pd.api.types.is_numeric_dtype(df[col_name]):
            ax.plot(time_axis, df[col_name].values, 
                   label=col_name, color=colors[i], linewidth=1.5, alpha=0.8)
    
    ax.set_xlabel(time_label, fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.set_title(group_name, fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plot_idx += 1

# Plot single columns
for col_name in single_columns:
    if col_name not in df.columns:
        continue
    
    row = plot_idx // n_cols
    col = plot_idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    
    if pd.api.types.is_numeric_dtype(df[col_name]):
        ax.plot(time_axis, df[col_name].values, 
               label=col_name, color='#1f77b4', linewidth=1.5, alpha=0.8)
        ax.set_ylabel('Value', fontsize=10)
    else:
        # For categorical data, create a scatter or step plot
        unique_vals = df[col_name].unique()
        if len(unique_vals) < 20:  # If few unique values, use step plot
            # Map categories to numbers for plotting
            cat_map = {val: i for i, val in enumerate(unique_vals)}
            numeric_values = df[col_name].map(cat_map)
            ax.step(time_axis, numeric_values.values, where='post', 
                   label=col_name, color='#1f77b4', linewidth=1.5, alpha=0.8)
            ax.set_yticks(range(len(unique_vals)))
            ax.set_yticklabels(unique_vals, fontsize=8)
            ax.set_ylabel('Category', fontsize=10)
        else:
            # Too many categories, just plot as numeric if possible
            try:
                numeric_vals = pd.to_numeric(df[col_name], errors='coerce')
                ax.plot(time_axis, numeric_vals.values, 
                       label=col_name, color='#1f77b4', linewidth=1.5, alpha=0.8)
                ax.set_ylabel('Value', fontsize=10)
            except:
                ax.text(0.5, 0.5, f'Cannot plot {col_name}\n(too many categories)', 
                       transform=ax.transAxes, ha='center', va='center')
    
    ax.set_xlabel(time_label, fontsize=10)
    ax.set_title(col_name, fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plot_idx += 1

plt.suptitle('All Columns vs Time', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

print(f"Created {plot_idx} plots showing all columns vs time")


In [ ]:
def fill_nan_with_previous(df, columns=None, inplace=False):
    """
    Fill NaN values with the previous non-NaN value (forward fill) in time.
    
    This function performs a forward fill operation, which propagates the last
    valid observation forward to fill NaN values. This is time-aware since the
    data is assumed to be ordered chronologically.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to process. Must be sorted by time.
    columns : list of str or None, optional
        List of column names to fill. If None, fills all columns.
        Default: None (fills all columns)
    inplace : bool, optional
        If True, modifies the dataframe in place. If False, returns a copy.
        Default: False
    
    Returns:
    --------
    pandas.DataFrame
        Dataframe with NaN values filled. If inplace=True, returns None.
    
    Examples:
    ---------
    >>> df_filled = fill_nan_with_previous(df)
    >>> fill_nan_with_previous(df, columns=['kite_0_vx', 'kite_0_vy'], inplace=True)
    """
    if inplace:
        result_df = df
    else:
        result_df = df.copy()
    
    # Determine which columns to process
    if columns is None:
        columns_to_fill = result_df.columns.tolist()
    else:
        # Filter to only existing columns
        columns_to_fill = [col for col in columns if col in result_df.columns]
        if len(columns_to_fill) == 0:
            print("Warning: None of the specified columns exist in the dataframe.")
            return result_df if not inplace else None
    
    # Count NaN values before filling
    nan_before = result_df[columns_to_fill].isna().sum().sum()
    
    # Perform forward fill (ffill) on selected columns
    result_df[columns_to_fill] = result_df[columns_to_fill].ffill()
    
    # Count NaN values after filling
    nan_after = result_df[columns_to_fill].isna().sum().sum()
    
    # Report results
    if nan_before > 0:
        print(f"Filled {nan_before - nan_after} NaN values")
        if nan_after > 0:
            print(f"Warning: {nan_after} NaN values remain (likely at the start of columns)")
            # Show which columns still have NaN
            remaining_nan = result_df[columns_to_fill].isna().sum()
            remaining_nan = remaining_nan[remaining_nan > 0]
            if len(remaining_nan) > 0:
                print("Columns with remaining NaN values:")
                for col, count in remaining_nan.items():
                    print(f"  {col}: {count} NaN values")
    else:
        print("No NaN values found to fill")
    
    return result_df if not inplace else None


In [ ]:
# Example usage:
# Fill all NaN values
df_filled = fill_nan_with_previous(df)

In [ ]:
# The batch cleaning function lives in functions/preprocessing/clean_csv.py
from functions.preprocessing.clean_csv import process_all_csvs

In [ ]:
dataset = '../data/dataset_1'
for split in ('training_data', 'test_data'):
    result = process_all_csvs(f'{dataset}/{split}', output_folder=f'{dataset}/cleaned/{split}', fill_nan=True)